<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/ORJ_DCASE_FEATURE_SELECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TARGET-AWARE GA FEATURE SELECTION FOR DCASE 2025 TASK 2
# Using 990 source-normal + 10 target-normal samples
# Search universe: 128 unique selected TDA features
# Detector inside GA: kNN distance
# Objective:
#   pseudo anomaly separation
#   target-normal acceptance
#   source-target normal alignment
#   source compactness
#   feature-count penalty
# ============================================================

import os
import glob
import json
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# GA parameters
POP_SIZE = 80
N_GENERATIONS = 60
ELITE_SIZE = 6
TOURNAMENT_SIZE = 3
CROSSOVER_PROB = 0.85
MUTATION_PROB = 0.08
BIT_FLIP_PROB = 0.03

# Feature subset size constraints
MIN_FEATURES = 20
MAX_FEATURES = 30

# kNN parameter
K_NEIGHBORS = 10

# Source threshold quantile for accepting target-normal samples
TARGET_ACCEPTANCE_QUANTILE = 0.99

# Pseudo anomaly generation
PSEUDO_MULTIPLIER = 1.0
PSEUDO_SHIFT_LOW = 1.5
PSEUDO_SHIFT_HIGH = 3.5
PSEUDO_CORRUPT_RATIO = 0.25

# Target-aware objective weights
# Target effect = target_acceptance + target_alignment = 0.50
WEIGHTS = {
    "pseudo_auc": 0.30,
    "target_acceptance": 0.35,
    "target_alignment": 0.15,
    "source_compactness": 0.15,
    "feature_penalty": 0.05,
}

MACHINES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]


# ============================================================
# 2. INITIAL 128 UNIQUE FEATURE SET
# ============================================================

initial_128_features = [
    "H0_Carlsson_f3",
    "H0_betti_argmax",
    "H0_betti_entropy",
    "H0_betti_l1",
    "H0_betti_mean",
    "H0_betti_num_peaks",
    "H0_birth_kurtosis",
    "H0_birth_max",
    "H0_birth_min",
    "H0_birth_skew",
    "H0_death_kurtosis",
    "H0_death_median",
    "H0_death_min",
    "H0_entropy_lifetime_ratio",
    "H0_iqr_lifetime",
    "H0_l2_norm_lifetime",
    "H0_landscape_entropy",
    "H0_landscape_layer1_max",
    "H0_landscape_layer2_max",
    "H0_landscape_layer2_mean",
    "H0_landscape_layer3_max",
    "H0_landscape_layer3_mean",
    "H0_landscape_layer4_max",
    "H0_landscape_layer4_mean",
    "H0_landscape_layer5_auc",
    "H0_landscape_layer5_max",
    "H0_landscape_mean",
    "H0_landscape_var",
    "H0_lifetime_energy",
    "H0_lifetime_skew",
    "H0_max_diag_distance",
    "H0_max_lifetime",
    "H0_mean_diag_distance",
    "H0_mean_lifetime",
    "H0_normalized_persistence_entropy",
    "H0_num_points",
    "H0_persistence_entropy",
    "H0_pimage_entropy",
    "H0_pimage_l1",
    "H0_pimage_max",
    "H0_pimage_min",
    "H0_pimage_std",
    "H0_q10_lifetime",
    "H0_q25_lifetime",
    "H0_q95_lifetime",
    "H0_range_lifetime",
    "H0_silhouette_auc",
    "H0_silhouette_entropy",
    "H0_sum_lifetime",
    "H0_tail_share_q95",
    "H0_top1_share",
    "H0_top3_share",
    "H0_top5_share",
    "H0_weighted_death_std",

    "H1_betti_energy",
    "H1_betti_entropy",
    "H1_betti_l1",
    "H1_betti_l2",
    "H1_betti_max",
    "H1_betti_mean",
    "H1_betti_num_peaks",
    "H1_betti_std",
    "H1_betti_var",
    "H1_birth_kurtosis",
    "H1_birth_max",
    "H1_birth_median",
    "H1_birth_q25",
    "H1_birth_q75",
    "H1_birth_skew",
    "H1_birth_std",
    "H1_death_kurtosis",
    "H1_death_max",
    "H1_death_median",
    "H1_death_skew",
    "H1_death_std",
    "H1_entropy_lifetime_ratio",
    "H1_iqr_lifetime",
    "H1_l1_norm_lifetime",
    "H1_l2_norm_lifetime",
    "H1_landscape_entropy",
    "H1_landscape_l1",
    "H1_landscape_layer1_auc",
    "H1_landscape_layer2_auc",
    "H1_landscape_layer2_max",
    "H1_landscape_layer3_max",
    "H1_landscape_layer5_max",
    "H1_landscape_layer5_mean",
    "H1_landscape_std",
    "H1_lifetime_energy",
    "H1_lifetime_rms",
    "H1_linf_norm_lifetime",
    "H1_mean_birth_death_ratio",
    "H1_min_lifetime",
    "H1_min_midlife",
    "H1_num_points",
    "H1_persistence_entropy",
    "H1_pimage_energy",
    "H1_pimage_entropy",
    "H1_pimage_l2",
    "H1_pimage_max",
    "H1_pimage_std",
    "H1_pimage_var",
    "H1_points_raw",
    "H1_q10_lifetime",
    "H1_q90_lifetime",
    "H1_q95_lifetime",
    "H1_range_lifetime",
    "H1_silhouette_entropy",
    "H1_silhouette_l2",
    "H1_silhouette_max",
    "H1_silhouette_std",
    "H1_silhouette_var",
    "H1_std_midlife",
    "H1_sum_diag_distance",
    "H1_sum_lifetime",
    "H1_tail_share_q80",
    "H1_tail_share_q90",
    "H1_tail_share_q95",
    "H1_to_H0_energy_ratio",
    "H1_to_H0_entropy_ratio",
    "H1_to_H0_max_lifetime_ratio",
    "H1_to_H0_num_points_ratio",
    "H1_to_H0_sum_lifetime_ratio",
    "H1_top1_share",
    "H1_top5_share",
    "H1_weighted_death_mean",
    "H1_weighted_death_std",
    "H1_weighted_midlife_mean",
]

initial_128_features = sorted(list(dict.fromkeys(initial_128_features)))

print("Number of initial unique features:", len(initial_128_features))


# ============================================================
# 3. FILE DISCOVERY
# ============================================================

def find_machine_file(machine, search_dirs=("/content", "/mnt/data", ".")):
    """
    Finds the threshold-training Excel file for a machine.
    Expected examples:
        cubical_mel_tda_features_AutoTrash_thr(3).xlsx
        cubical_mel_tda_features_BandSealer_thr(1).xlsx
    """
    patterns = []
    for d in search_dirs:
        patterns.extend([
            os.path.join(d, f"cubical_mel_tda_features_{machine}_thr*.xlsx"),
            os.path.join(d, f"*{machine}*thr*.xlsx"),
        ])

    matches = []
    for p in patterns:
        matches.extend(glob.glob(p))

    matches = sorted(list(set(matches)))

    if len(matches) == 0:
        raise FileNotFoundError(f"No threshold-training Excel file found for machine: {machine}")

    return matches[0]


def detect_source_target_split(df):
    """
    Detects source-normal and target-normal samples.

    In these datasets, the FIRST COLUMN contains the file name/path,
    and this name includes either 'source' or 'target'.

    Example values:
        section_00_source_train_normal_0001.wav
        section_00_target_train_normal_0001.wav

    Therefore:
        rows where first-column value contains 'source' -> source normal
        rows where first-column value contains 'target' -> target normal
    """

    first_col = df.columns[0]

    values = df[first_col].astype(str).str.lower()

    source_mask = values.str.contains("source", na=False)
    target_mask = values.str.contains("target", na=False)

    n_source = int(source_mask.sum())
    n_target = int(target_mask.sum())

    print(f"First column used for domain detection: {first_col}")
    print(f"Detected source rows: {n_source}")
    print(f"Detected target rows: {n_target}")

    if n_source == 0 or n_target == 0:
        raise ValueError(
            f"Could not detect both source and target samples from first column: {first_col}. "
            "Please check whether the first column values contain 'source' or 'target'."
        )

    source_df = df.loc[source_mask].copy()
    target_df = df.loc[target_mask].copy()

    return source_df, target_df, first_col


# ============================================================
# 5. kNN DISTANCE SCORING
# ============================================================

def compute_knn_scores(X_source, X_query, k=10, query_is_source=False):
    """
    Higher score = more anomalous.

    If query_is_source=True:
        uses k+1 neighbors and removes self-neighbor.
    """

    n_source = X_source.shape[0]
    k_eff = min(k, n_source - 1)

    if k_eff < 1:
        raise ValueError("Not enough source samples for kNN scoring.")

    if query_is_source:
        nn = NearestNeighbors(n_neighbors=k_eff + 1, metric="euclidean")
        nn.fit(X_source)
        distances, _ = nn.kneighbors(X_source)
        distances = distances[:, 1:]  # remove self-neighbor
    else:
        nn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean")
        nn.fit(X_source)
        distances, _ = nn.kneighbors(X_query)

    scores = distances.mean(axis=1)
    return scores


# ============================================================
# 6. PSEUDO ANOMALY GENERATION
# ============================================================

def make_pseudo_anomalies(X_source_scaled, n_pseudo, rng):
    """
    Creates pseudo anomalies from source-normal data.

    Three corruption mechanisms:
        1. feature permutation
        2. random feature shifts
        3. extreme-tail replacement

    Input should already be scaled.
    """

    n, d = X_source_scaled.shape

    if d == 0:
        raise ValueError("No features selected.")

    n_each = int(np.ceil(n_pseudo / 3))

    # --------------------------------------------------------
    # 1. Permutation corruption
    # --------------------------------------------------------
    idx = rng.choice(n, size=n_each, replace=True)
    X_perm = X_source_scaled[idx].copy()

    for j in range(d):
        rng.shuffle(X_perm[:, j])

    # --------------------------------------------------------
    # 2. Random shift corruption
    # --------------------------------------------------------
    idx = rng.choice(n, size=n_each, replace=True)
    X_shift = X_source_scaled[idx].copy()

    n_corrupt_features = max(1, int(np.ceil(PSEUDO_CORRUPT_RATIO * d)))

    for i in range(n_each):
        cols = rng.choice(d, size=n_corrupt_features, replace=False)
        signs = rng.choice([-1.0, 1.0], size=n_corrupt_features)
        shifts = rng.uniform(PSEUDO_SHIFT_LOW, PSEUDO_SHIFT_HIGH, size=n_corrupt_features)
        X_shift[i, cols] += signs * shifts

    # --------------------------------------------------------
    # 3. Extreme-tail replacement
    # --------------------------------------------------------
    idx = rng.choice(n, size=n_each, replace=True)
    X_extreme = X_source_scaled[idx].copy()

    q01 = np.nanquantile(X_source_scaled, 0.01, axis=0)
    q99 = np.nanquantile(X_source_scaled, 0.99, axis=0)

    for i in range(n_each):
        cols = rng.choice(d, size=n_corrupt_features, replace=False)
        high_or_low = rng.choice([0, 1], size=n_corrupt_features)

        for c_idx, col in enumerate(cols):
            if high_or_low[c_idx] == 1:
                X_extreme[i, col] = q99[col] + rng.uniform(0.5, 2.0)
            else:
                X_extreme[i, col] = q01[col] - rng.uniform(0.5, 2.0)

    X_pseudo = np.vstack([X_perm, X_shift, X_extreme])
    X_pseudo = X_pseudo[:n_pseudo]

    return X_pseudo


# ============================================================
# 7. PREPROCESSING FOR A FEATURE SUBSET
# ============================================================

def prepare_selected_data(source_df, target_df, selected_features):
    """
    Fits imputer and scaler on source-normal only.
    Transforms source and target.
    Drops constant selected features using source data.
    """

    Xs_raw = source_df[selected_features].replace([np.inf, -np.inf], np.nan).values
    Xt_raw = target_df[selected_features].replace([np.inf, -np.inf], np.nan).values

    # Drop features that are fully NaN in source
    valid_not_all_nan = ~np.all(np.isnan(Xs_raw), axis=0)

    if valid_not_all_nan.sum() == 0:
        return None, None, []

    Xs_raw = Xs_raw[:, valid_not_all_nan]
    Xt_raw = Xt_raw[:, valid_not_all_nan]
    kept_features = [f for f, keep in zip(selected_features, valid_not_all_nan) if keep]

    # Impute missing values using source medians
    imputer = SimpleImputer(strategy="median")
    Xs_imp = imputer.fit_transform(Xs_raw)
    Xt_imp = imputer.transform(Xt_raw)

    # Drop near-constant features after imputation
    stds = np.std(Xs_imp, axis=0)
    non_constant = stds > 1e-12

    if non_constant.sum() == 0:
        return None, None, []

    Xs_imp = Xs_imp[:, non_constant]
    Xt_imp = Xt_imp[:, non_constant]
    kept_features = [f for f, keep in zip(kept_features, non_constant) if keep]

    # Robust scaling fitted on source only
    scaler = RobustScaler()
    Xs_scaled = scaler.fit_transform(Xs_imp)
    Xt_scaled = scaler.transform(Xt_imp)

    Xs_scaled = np.nan_to_num(Xs_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    Xt_scaled = np.nan_to_num(Xt_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    return Xs_scaled, Xt_scaled, kept_features


# ============================================================
# 8. FITNESS FUNCTION
# ============================================================

def evaluate_subset(
    mask,
    source_df,
    target_df,
    feature_names,
    rng,
    return_details=False,
):
    """
    Evaluates one GA individual.

    Objective components:
        1. pseudo anomaly AUC
        2. target-normal acceptance
        3. target alignment
        4. source compactness
        5. feature-count penalty
    """

    mask = np.array(mask).astype(bool)
    n_selected = int(mask.sum())

    if n_selected < MIN_FEATURES or n_selected > MAX_FEATURES:
        if return_details:
            return {
                "fitness": -1e9,
                "n_features": n_selected,
                "reason": "feature_count_out_of_bounds",
            }
        return -1e9

    selected_features = [f for f, keep in zip(feature_names, mask) if keep]

    try:
        Xs, Xt, kept_features = prepare_selected_data(
            source_df=source_df,
            target_df=target_df,
            selected_features=selected_features,
        )

        if Xs is None or Xt is None or len(kept_features) < MIN_FEATURES:
            if return_details:
                return {
                    "fitness": -1e9,
                    "n_features": len(kept_features),
                    "reason": "too_few_valid_features",
                }
            return -1e9

        # ----------------------------------------------------
        # Source and target kNN scores
        # ----------------------------------------------------
        source_scores = compute_knn_scores(
            X_source=Xs,
            X_query=Xs,
            k=K_NEIGHBORS,
            query_is_source=True,
        )

        target_scores = compute_knn_scores(
            X_source=Xs,
            X_query=Xt,
            k=K_NEIGHBORS,
            query_is_source=False,
        )

        source_q50 = np.quantile(source_scores, 0.50)
        source_q75 = np.quantile(source_scores, 0.75)
        source_q95 = np.quantile(source_scores, 0.95)
        source_q99 = np.quantile(source_scores, TARGET_ACCEPTANCE_QUANTILE)

        eps = 1e-12

        # ----------------------------------------------------
        # Target acceptance score
        # ----------------------------------------------------
        target_hard_acceptance = np.mean(target_scores <= source_q99)

        target_excess = np.maximum(0.0, target_scores - source_q99)
        normalizer = max(source_q99 - source_q50, eps)

        target_soft_penalty = np.mean(target_excess / normalizer)
        target_soft_acceptance = 1.0 / (1.0 + target_soft_penalty)

        target_acceptance = (
            0.60 * target_hard_acceptance
            + 0.40 * target_soft_acceptance
        )

        # ----------------------------------------------------
        # Target alignment score
        # Penalizes target distances only when they become
        # larger than the normal source distribution.
        # ----------------------------------------------------
        target_median = np.median(target_scores)
        target_q90 = np.quantile(target_scores, 0.90)

        median_excess = max(0.0, target_median - source_q50) / max(source_q75 - source_q50, eps)
        q90_excess = max(0.0, target_q90 - source_q95) / max(source_q95 - source_q50, eps)

        target_alignment = 1.0 / (1.0 + median_excess + q90_excess)

        # ----------------------------------------------------
        # Source compactness score
        # Compact source distribution is preferred.
        # ----------------------------------------------------
        source_spread = (source_q95 - source_q50) / max(source_q95, eps)
        source_spread = np.clip(source_spread, 0.0, 1.0)

        source_compactness = 1.0 - source_spread

        # ----------------------------------------------------
        # Pseudo anomaly AUC
        # ----------------------------------------------------
        n_pseudo = int(PSEUDO_MULTIPLIER * len(Xs))
        X_pseudo = make_pseudo_anomalies(Xs, n_pseudo=n_pseudo, rng=rng)

        pseudo_scores = compute_knn_scores(
            X_source=Xs,
            X_query=X_pseudo,
            k=K_NEIGHBORS,
            query_is_source=False,
        )

        y_auc = np.r_[np.zeros(len(source_scores)), np.ones(len(pseudo_scores))]
        s_auc = np.r_[source_scores, pseudo_scores]

        pseudo_auc = roc_auc_score(y_auc, s_auc)

        # If pseudo anomalies are not ranked above source normals,
        # keep the true AUC value. Do not reverse it.
        pseudo_auc = float(pseudo_auc)

        # ----------------------------------------------------
        # Feature-count penalty
        # ----------------------------------------------------
        feature_penalty = len(kept_features) / float(MAX_FEATURES)

        # ----------------------------------------------------
        # Final target-aware fitness
        # ----------------------------------------------------
        fitness = (
            WEIGHTS["pseudo_auc"] * pseudo_auc
            + WEIGHTS["target_acceptance"] * target_acceptance
            + WEIGHTS["target_alignment"] * target_alignment
            + WEIGHTS["source_compactness"] * source_compactness
            - WEIGHTS["feature_penalty"] * feature_penalty
        )

        if return_details:
            return {
                "fitness": float(fitness),
                "pseudo_auc": float(pseudo_auc),
                "target_acceptance": float(target_acceptance),
                "target_hard_acceptance": float(target_hard_acceptance),
                "target_soft_acceptance": float(target_soft_acceptance),
                "target_alignment": float(target_alignment),
                "source_compactness": float(source_compactness),
                "feature_penalty": float(feature_penalty),
                "n_features": int(len(kept_features)),
                "selected_features": kept_features,
                "source_score_q50": float(source_q50),
                "source_score_q95": float(source_q95),
                "source_score_q99": float(source_q99),
                "target_score_mean": float(np.mean(target_scores)),
                "target_score_max": float(np.max(target_scores)),
                "target_scores": target_scores.tolist(),
                "reason": "ok",
            }

        return float(fitness)

    except Exception as e:
        if return_details:
            return {
                "fitness": -1e9,
                "n_features": n_selected,
                "reason": f"error: {str(e)}",
            }
        return -1e9


# ============================================================
# 9. GA OPERATORS
# ============================================================

def repair_mask(mask, rng):
    """
    Ensures feature count is between MIN_FEATURES and MAX_FEATURES.
    """

    mask = np.array(mask).astype(bool)
    n = len(mask)
    count = int(mask.sum())

    if count > MAX_FEATURES:
        selected_idx = np.where(mask)[0]
        remove_count = count - MAX_FEATURES
        remove_idx = rng.choice(selected_idx, size=remove_count, replace=False)
        mask[remove_idx] = False

    elif count < MIN_FEATURES:
        not_selected_idx = np.where(~mask)[0]
        add_count = MIN_FEATURES - count
        add_idx = rng.choice(not_selected_idx, size=add_count, replace=False)
        mask[add_idx] = True

    return mask


def random_individual(n_features, rng):
    """
    Random subset selected from the 128-feature universe.
    """

    mask = np.zeros(n_features, dtype=bool)
    n_selected = rng.integers(MIN_FEATURES, MAX_FEATURES + 1)
    selected_idx = rng.choice(n_features, size=n_selected, replace=False)
    mask[selected_idx] = True
    return mask


def tournament_selection(population, fitnesses, rng):
    """
    Tournament selection.
    """

    idxs = rng.choice(len(population), size=TOURNAMENT_SIZE, replace=False)
    best_idx = idxs[np.argmax([fitnesses[i] for i in idxs])]
    return population[best_idx].copy()


def uniform_crossover(parent1, parent2, rng):
    """
    Uniform crossover.
    """

    if rng.random() > CROSSOVER_PROB:
        return parent1.copy(), parent2.copy()

    mask = rng.random(len(parent1)) < 0.5

    child1 = parent1.copy()
    child2 = parent2.copy()

    child1[mask] = parent2[mask]
    child2[mask] = parent1[mask]

    child1 = repair_mask(child1, rng)
    child2 = repair_mask(child2, rng)

    return child1, child2


def mutate(mask, rng):
    """
    Bit-flip mutation plus repair.
    """

    child = mask.copy()

    if rng.random() < MUTATION_PROB:
        flip_mask = rng.random(len(child)) < BIT_FLIP_PROB
        child[flip_mask] = ~child[flip_mask]

    child = repair_mask(child, rng)
    return child


# ============================================================
# 10. RUN GA FOR ONE MACHINE
# ============================================================

def run_ga_for_machine(machine, df, available_features):
    """
    Runs GA for one machine and returns selected features.
    """

    print("\n" + "=" * 100)
    print(f"RUNNING TARGET-AWARE GA FOR MACHINE: {machine}")
    print("=" * 100)

    source_df, target_df, domain_col = detect_source_target_split(df)

    print(f"Available candidate features: {len(available_features)}")
    print(f"Source normal samples: {len(source_df)}")
    print(f"Target normal samples: {len(target_df)}")

    rng = np.random.default_rng(RANDOM_SEED + abs(hash(machine)) % 100000)

    n_features = len(available_features)

    # Initialize population
    population = [random_individual(n_features, rng) for _ in range(POP_SIZE)]

    # Optional seed: one individual with exactly MAX_FEATURES random features
    seed_mask = np.zeros(n_features, dtype=bool)
    seed_idx = rng.choice(n_features, size=MAX_FEATURES, replace=False)
    seed_mask[seed_idx] = True
    population[0] = seed_mask

    fitness_cache = {}

    def cached_fitness(mask):
        key = tuple(mask.astype(int).tolist())
        if key not in fitness_cache:
            fitness_cache[key] = evaluate_subset(
                mask=mask,
                source_df=source_df,
                target_df=target_df,
                feature_names=available_features,
                rng=rng,
                return_details=False,
            )
        return fitness_cache[key]

    best_history = []

    for gen in range(1, N_GENERATIONS + 1):

        fitnesses = np.array([cached_fitness(ind) for ind in population])

        ranked_idx = np.argsort(fitnesses)[::-1]
        population = [population[i] for i in ranked_idx]
        fitnesses = fitnesses[ranked_idx]

        best_fit = fitnesses[0]
        mean_fit = np.mean(fitnesses[np.isfinite(fitnesses)])

        best_history.append(best_fit)

        if gen == 1 or gen % 10 == 0 or gen == N_GENERATIONS:
            best_count = int(population[0].sum())
            print(
                f"Generation {gen:03d} | "
                f"Best fitness: {best_fit:.5f} | "
                f"Mean fitness: {mean_fit:.5f} | "
                f"Selected features: {best_count}"
            )

        # Elitism
        new_population = [population[i].copy() for i in range(ELITE_SIZE)]

        # Generate rest of population
        while len(new_population) < POP_SIZE:
            p1 = tournament_selection(population, fitnesses, rng)
            p2 = tournament_selection(population, fitnesses, rng)

            c1, c2 = uniform_crossover(p1, p2, rng)

            c1 = mutate(c1, rng)
            c2 = mutate(c2, rng)

            new_population.append(c1)

            if len(new_population) < POP_SIZE:
                new_population.append(c2)

        population = new_population

    # Final evaluation
    fitnesses = np.array([cached_fitness(ind) for ind in population])
    best_idx = int(np.argmax(fitnesses))
    best_mask = population[best_idx]

    best_details = evaluate_subset(
        mask=best_mask,
        source_df=source_df,
        target_df=target_df,
        feature_names=available_features,
        rng=rng,
        return_details=True,
    )

    print("\n" + "-" * 100)
    print(f"BEST RESULT FOR {machine}")
    print("-" * 100)
    print(f"Fitness                  : {best_details['fitness']:.5f}")
    print(f"Pseudo anomaly AUC        : {best_details['pseudo_auc']:.5f}")
    print(f"Target acceptance         : {best_details['target_acceptance']:.5f}")
    print(f"Target hard acceptance    : {best_details['target_hard_acceptance']:.5f}")
    print(f"Target alignment          : {best_details['target_alignment']:.5f}")
    print(f"Source compactness        : {best_details['source_compactness']:.5f}")
    print(f"Number of features        : {best_details['n_features']}")
    print(f"Source q99 threshold      : {best_details['source_score_q99']:.5f}")
    print(f"Target score max          : {best_details['target_score_max']:.5f}")

    print("\nSelected features:")
    for f in best_details["selected_features"]:
        print(f"- {f}")

    best_details["machine"] = machine
    best_details["domain_col"] = domain_col
    best_details["best_history"] = best_history

    return best_details


# ============================================================
# 11. MAIN EXECUTION
# ============================================================

selected_features_by_machine = {}
summary_rows = []
details_by_machine = {}

for machine in MACHINES:

    file_path = find_machine_file(machine)
    print(f"\nReading file for {machine}: {file_path}")

    df = pd.read_excel(file_path)

    # Keep only features that exist in this machine file
    available_features = [f for f in initial_128_features if f in df.columns]
    missing_features = [f for f in initial_128_features if f not in df.columns]

    print(f"Total columns in file: {df.shape[1]}")
    print(f"Rows in file: {df.shape[0]}")
    print(f"Available initial features: {len(available_features)}")
    print(f"Missing initial features: {len(missing_features)}")

    if len(available_features) < MIN_FEATURES:
        raise ValueError(
            f"Machine {machine} has only {len(available_features)} available features. "
            f"Minimum required is {MIN_FEATURES}."
        )

    best_details = run_ga_for_machine(
        machine=machine,
        df=df,
        available_features=available_features,
    )

    selected_features_by_machine[machine] = best_details["selected_features"]
    details_by_machine[machine] = best_details

    summary_rows.append({
        "machine": machine,
        "fitness": best_details["fitness"],
        "pseudo_auc": best_details["pseudo_auc"],
        "target_acceptance": best_details["target_acceptance"],
        "target_hard_acceptance": best_details["target_hard_acceptance"],
        "target_alignment": best_details["target_alignment"],
        "source_compactness": best_details["source_compactness"],
        "feature_penalty": best_details["feature_penalty"],
        "n_features": best_details["n_features"],
        "source_score_q50": best_details["source_score_q50"],
        "source_score_q95": best_details["source_score_q95"],
        "source_score_q99": best_details["source_score_q99"],
        "target_score_mean": best_details["target_score_mean"],
        "target_score_max": best_details["target_score_max"],
    })


# ============================================================
# 12. PRINT FINAL DICTIONARY
# ============================================================

print("\n\n" + "#" * 100)
print("FINAL GA-SELECTED FEATURE DICTIONARY")
print("#" * 100)

print("selected_features_by_machine = {")
for machine, feats in selected_features_by_machine.items():
    print(f"    {repr(machine)}: [")
    for f in feats:
        print(f"        {repr(f)},")
    print("    ],")
print("}")


# ============================================================
# 13. SAVE OUTPUT FILES
# ============================================================

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("machine").reset_index(drop=True)

summary_csv_path = "target_aware_ga_feature_selection_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

json_path = "target_aware_ga_selected_features_by_machine.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(selected_features_by_machine, f, indent=4)

py_path = "target_aware_ga_selected_features_by_machine.py"
with open(py_path, "w", encoding="utf-8") as f:
    f.write("selected_features_by_machine = {\n")
    for machine, feats in selected_features_by_machine.items():
        f.write(f"    {repr(machine)}: [\n")
        for feat in feats:
            f.write(f"        {repr(feat)},\n")
        f.write("    ],\n")
    f.write("}\n")

details_path = "target_aware_ga_full_details.json"
details_to_save = {}

for machine, details in details_by_machine.items():
    clean_details = {}
    for k, v in details.items():
        if isinstance(v, np.ndarray):
            clean_details[k] = v.tolist()
        elif isinstance(v, (np.float32, np.float64)):
            clean_details[k] = float(v)
        elif isinstance(v, (np.int32, np.int64)):
            clean_details[k] = int(v)
        else:
            clean_details[k] = v
    details_to_save[machine] = clean_details

with open(details_path, "w", encoding="utf-8") as f:
    json.dump(details_to_save, f, indent=4)

print("\nSaved files:")
print(f"- {summary_csv_path}")
print(f"- {json_path}")
print(f"- {py_path}")
print(f"- {details_path}")

print("\nSummary:")
display(summary_df)